# FlowCast cleaning and validation

This notebook explains the reproducible cleaning pipeline. Original source files remain unchanged. Run `python -m src.utils --stage data` from the project root to regenerate the processed data and audit.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from IPython.display import display, Image
from src.features import load_dataset
from src.utils import ROOT
import json
print(json.dumps(json.loads((ROOT / 'reports/data_quality.json').read_text()), indent=2))
df = load_dataset()

{
  "source_rows": {
    "calendar_events": 151,
    "weather_observations": 10872,
    "traffic_sensor_log": 178468
  },
  "duplicates": 1767,
  "weather_duplicates": 0,
  "calendar_duplicates": 0,
  "absent_windows": 4499,
  "processed_rows": 181200,
  "missing_sensor_cells_after_validation": {
    "traffic_volume": 9086,
    "avg_speed": 9078,
    "occupancy": 9072
  },
  "statistical_extremes_retained": {
    "traffic_volume": 124,
    "avg_speed": 0,
    "occupancy": 0
  },
  "audit_records": 5834,
  "weather_missing_cells": {
    "temperature": 167,
    "visibility": 111,
    "rainfall": 0
  },
  "blank_congestion_source_rows": 26883
}


## Validation and duplicates

The validator checks keys, date formats, ranges and vehicle-share JSON. Invalid numeric cells are marked missing. Unrecoverable records are quarantined. Duplicate keys retain the most complete row. The audit keeps the original CSV row number.

In [2]:
audit = pd.read_csv(ROOT / 'data/interim/validation_audit.csv')
display(audit.groupby(['field', 'reason', 'action']).size().reset_index(name='rows'))
display(audit.head(8))

,field,reason,action,rows
0,avg_speed,Invalid numeric type or physical range,set cell missing,237
1,occupancy,Invalid numeric type or physical range,set cell missing,234
2,road_id+timestamp,Duplicate key; most complete record retained,remove duplicate,1767
3,traffic_volume,Invalid numeric type or physical range,set cell missing,241
4,weather_condition,Noncanonical valid weather label,normalize category,3355


,table,source_row,field,original_value,reason,action
0,weather_observations,4,weather_condition,clear,Noncanonical valid weather label,normalize category
1,weather_observations,6,weather_condition,clear,Noncanonical valid weather label,normalize category
2,weather_observations,7,weather_condition,clear,Noncanonical valid weather label,normalize category
3,weather_observations,11,weather_condition,cloudy,Noncanonical valid weather label,normalize category
4,weather_observations,12,weather_condition,Overcast,Noncanonical valid weather label,normalize category
5,weather_observations,14,weather_condition,Overcast,Noncanonical valid weather label,normalize category
6,weather_observations,15,weather_condition,Overcast,Noncanonical valid weather label,normalize category
7,weather_observations,16,weather_condition,cloudy,Noncanonical valid weather label,normalize category


## Missing windows and causal values

The complete road-by-time grid identifies absent readings. Historical interpolation is for analysis only. Model features use the causal columns, which never read future values. Accident and travel-time targets are not imputed. Observation flags prevent repaired outcomes from entering test scores.

In [3]:
display(pd.read_csv(ROOT / 'data/interim/missing_windows.csv').head(8))
display(df.loc[~df.traffic_volume_observed, ['road_id', 'timestamp', 'traffic_volume', 'causal_traffic_volume', 'traffic_volume_observed']].head(8))
assert not df.duplicated(['road_id', 'timestamp']).any()
assert len(df) == 25 * 151 * 48
print('Grid keys and row-count checks passed.')

,road_id,timestamp
0,NL-001,2025-01-01 04:00:00+05:30
1,NL-001,2025-01-02 02:00:00+05:30
2,NL-001,2025-01-02 03:30:00+05:30
3,NL-001,2025-01-03 09:00:00+05:30
4,NL-001,2025-01-04 01:30:00+05:30
5,NL-001,2025-01-05 00:00:00+05:30
6,NL-001,2025-01-05 16:30:00+05:30
7,NL-001,2025-01-05 21:00:00+05:30


,road_id,timestamp,traffic_volume,causal_traffic_volume,traffic_volume_observed
8,NL-001,2025-01-01 04:00:00+05:30,97.5,91.0,False
14,NL-001,2025-01-01 07:00:00+05:30,357.0,235.0,False
48,NL-001,2025-01-02 00:00:00+05:30,137.0,100.0,False
52,NL-001,2025-01-02 02:00:00+05:30,183.5,164.0,False
55,NL-001,2025-01-02 03:30:00+05:30,170.0,150.0,False
58,NL-001,2025-01-02 05:00:00+05:30,205.5,177.0,False
80,NL-001,2025-01-02 16:00:00+05:30,644.5,560.0,False
97,NL-001,2025-01-03 00:30:00+05:30,182.0,177.0,False


Grid keys and row-count checks passed.


## Joins and congestion

Weather dates are DD/MM/YYYY, while sensor and calendar dates are YYYY-MM-DD. Each hourly weather record joins both half-hour sensor windows using its station. Calendar flags join by local date. Missing congestion labels use volume / (capacity / 2), with class boundaries 0.50, 0.80 and 1.00. Asia/Kolkata was confirmed by the user.

In [4]:
display(df[['road_id', 'timestamp', 'weather_condition', 'rainfall', 'public_holiday', 'congestion_level']].head(8))
print((ROOT / 'reports/data_quality.md').read_text())

,road_id,timestamp,weather_condition,rainfall,public_holiday,congestion_level
0,NL-001,2025-01-01 00:00:00+05:30,Clear,0.0,1,Free-flow
1,NL-001,2025-01-01 00:30:00+05:30,Clear,0.0,1,Free-flow
2,NL-001,2025-01-01 01:00:00+05:30,Clear,0.0,1,Free-flow
3,NL-001,2025-01-01 01:30:00+05:30,Clear,0.0,1,Free-flow
4,NL-001,2025-01-01 02:00:00+05:30,Clear,0.0,1,Free-flow
5,NL-001,2025-01-01 02:30:00+05:30,Clear,0.0,1,Free-flow
6,NL-001,2025-01-01 03:00:00+05:30,Clear,0.0,1,Free-flow
7,NL-001,2025-01-01 03:30:00+05:30,Clear,0.0,1,Free-flow


# FlowCast data quality report

| Measure | Count |
| --- | --- |
| Traffic input rows | 178468 |
| Traffic duplicates | 1767 |
| Absent windows | 4499 |
| Processed rows | 181200 |
| Roads | 25 |
| Validation audit entries | 5834 |

## Source preservation and validation

The three raw CSV files are byte-for-byte copies of the supplied files. Their SHA-256 hashes
are in `data/processed/source_manifest.json`. The original Word files remain in their source
folder and are unchanged. Dates use Asia/Kolkata, confirmed by the user. Traffic and calendar
dates use YYYY-MM-DD. Weather dates use DD/MM/YYYY.

The pipeline checks columns, keys, JSON vehicle shares, categories, numeric types and physical
ranges. Invalid cells become missing and are listed by original CSV row in
`data/interim/validation_audit.csv`. Unrecoverable rows are quarantined with their full contents.
Duplicate keys keep the most complete record, with original row order as the tie breaker.

## Repairs and their limits

The re